In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

column_names = ['scenario',
                'episode',
                'ped_distance',
                'ped_speed',
                'collision',
                'nearmiss',
                'goal',
                'ttg',
                'total_episode_reward',
                'travelled_distance',
                'execution_times',
                'is_ped_observable',
                'car_speeds']
    
list_columns = ["is_ped_observable", 'execution_times', 'car_speeds']

stats_dir ="/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/"

scenario_performance_dfs = []
json_files = glob.glob(stats_dir + "*.json")
for json_file_path in json_files:
    print(json_file_path)
    with open(json_file_path, "r") as file:
        file.readline()
        scenario_df = pd.read_json(file, lines=True)
        print(scenario_df.shape)
        scenario_performance_dfs.append(scenario_df)

benchmark_performance_df = scenario_performance_dfs[0]
for scenario_df in scenario_performance_dfs[1:]:
    benchmark_performance_df = pd.concat([benchmark_performance_df, scenario_df], axis="index")
    print(benchmark_performance_df.shape)

/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-06_01.12.2024_16.20.21.json
(1241, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-08_30.11.2024_15.43.21.json
(34, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-06_30.11.2024_15.41.05.json
(51, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-05_30.11.2024_15.40.23.json
(47, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-05_01.12.2024_16.20.20.json
(1241, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-08_26.11.2024_08.20.57.json
(1241, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-02_01.12.2024_16.26.57.json
(1241, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_metrics_S-08_01.12.2024_16.21.17.json
(1241, 15)
/home/dopf02-admin/Carla-CTS02_backup/logs/metrics/test/performance_me

In [77]:
for column in benchmark_performance_df: print(column)

scenario
episode
ped_distance
ped_speed
collision
nearmiss
goal
ttg
total_episode_reward
travelled_distance
execution_times
non_conclusive
skipped_steps


In [2]:
# drop non-conclusive episodes
print("benchmark_performance_df.shape: ", benchmark_performance_df.shape)
num_non_conclusive = (benchmark_performance_df["non_conclusive"] == 1).sum()
print(f"number of non-conclusive episodes: {num_non_conclusive} ({(num_non_conclusive/len(benchmark_performance_df))*100:.2f}%)")
# benchmark_performance_df = benchmark_performance_df[benchmark_performance_df["non_conclusive"] != 1]
meta_df = benchmark_performance_df[["scenario", "collision", "nearmiss", "goal", "ttg"]]
print(meta_df.describe()[["collision", "nearmiss", "goal", "ttg"]])

benchmark_performance_df.shape:  (22730, 15)


KeyError: 'non_conclusive'

In [2]:
execution_times_avg = benchmark_performance_df["execution_times"].map(lambda x: np.mean(x))\
                                                                .to_frame()\
                                                                .add_suffix("_avg")

execution_times_median = benchmark_performance_df["execution_times"].map(lambda x: np.median(x))\
                                                                    .to_frame()\
                                                                    .add_suffix("_median")

assert len(meta_df) == len(execution_times_avg)
assert len(meta_df) == len(execution_times_median)

performance_df = pd.concat(
    [meta_df, execution_times_avg, execution_times_median],
     axis="columns"
)
performance_df[["collision", "nearmiss", "goal"]] = performance_df[["collision", "nearmiss", "goal"]] * 100

performance_df.describe()[
    ["collision", "nearmiss", "goal", "ttg", "execution_times_avg", "execution_times_median"]
].apply(lambda s: s.apply('{0:.2f}'.format))

NameError: name 'benchmark_performance_df' is not defined

In [3]:
performance_df.groupby("scenario").count()

NameError: name 'performance_df' is not defined

In [4]:
scenario_df = performance_df.groupby("scenario").mean()
scenario_df["execution_times_median"] = performance_df.groupby("scenario").median()["execution_times_median"]
scenario_df

NameError: name 'performance_df' is not defined

In [5]:
collision_heatmap_df = benchmark_performance_df[['ped_distance', 'ped_speed', 'collision']]

# round pedestrian distance and speed to 2 decimal places
collision_heatmap_df['ped_distance'] = collision_heatmap_df['ped_distance'].round(2)
collision_heatmap_df['ped_speed'] = collision_heatmap_df['ped_speed'].round(2)

# group by pedestrian distanbce and speed, and calculate the mean collision rate
heatmap_data = (
    collision_heatmap_df.groupby(['ped_distance', 'ped_speed'])['collision']
    .mean()
    .unstack(fill_value=0)  # Pivot the data into a matrix
)

# plot the heatmap using Matplotlib
plt.figure(figsize=(10, 8))
plt.imshow(heatmap_data, cmap='YlGnBu', origin='lower', aspect='auto')

# ddd colorbar
cbar = plt.colorbar()
cbar.set_label('Average Collision Rate')

# set axis labels
plt.title('Heatmap of Collisions')
plt.xlabel('Pedestrian Velocity (m/s)')
plt.ylabel('Pedestrian Crossing Distance (m)')

# set tick positions and labels
plt.xticks(
    ticks=np.arange(len(heatmap_data.columns)),
    labels=heatmap_data.columns,
    rotation=90
)
plt.yticks(
    ticks=np.arange(len(heatmap_data.index)),
    labels=heatmap_data.index
)

plt.tight_layout()
plt.show()

NameError: name 'benchmark_performance_df' is not defined

In [6]:
collision_heatmap_df = benchmark_performance_df[['scenario', 'ped_distance', 'ped_speed', 'collision']]

# round pedestrian distance and speed to 2 decimal places
collision_heatmap_df['ped_distance'] = collision_heatmap_df['ped_distance'].round(2)
collision_heatmap_df['ped_speed'] = collision_heatmap_df['ped_speed'].round(2)

# unique scenario values
scenarios = sorted(collision_heatmap_df['scenario'].unique())

# create subplots: 3 rows x 4 columns for 12 scenarios
fig, axes = plt.subplots(3, 4, figsize=(20, 15), constrained_layout=True)

# flatten the axes array for easy indexing
axes = axes.flatten()

# iterate through scenarios and corresponding axes
for i, (scenario, ax) in enumerate(zip(scenarios, axes)):
    # filter for the current scenario
    scenario_df = collision_heatmap_df[collision_heatmap_df['scenario'] == scenario]

    # group by ped distance and speed, and calculate the mean collision rate
    heatmap_data = (
        scenario_df.groupby(['ped_distance', 'ped_speed'])['collision']
        .mean()
        .unstack(fill_value=0)  # Pivot the data into a matrix
    )

    # plot the heatmap
    im = ax.imshow(heatmap_data, cmap='YlGnBu', origin='lower', aspect='auto')

    # set the title for the subplot
    ax.set_title(f'Scenario {scenario}')
    ax.set_xlabel('Pedestrian Velocity (m/s)')
    ax.set_ylabel('Pedestrian Crossing Distance (m)')

    # set tick positions and labels
    ax.set_xticks(np.arange(len(heatmap_data.columns)))
    ax.set_xticklabels(heatmap_data.columns, rotation=90)
    ax.set_yticks(np.arange(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index)

# add a single colorbar for all subplots
cbar = fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.8, pad=0.02)
cbar.set_label('Average Collision Rate')

# hide unused subplots if scenarios are fewer than 12
for ax in axes[len(scenarios):]:
    ax.set_visible(False)

# display the plot
plt.suptitle('Heatmaps of Collisions for Each Scenario', fontsize=16)
plt.show()


NameError: name 'benchmark_performance_df' is not defined